# LoRAForge — Phase 1 on a free T4

This notebook runs only the frozen first half: real development data, untuned validation baseline, and QLoRA adapter setup. **Do not load the publisher test split here.** Select the adapter and calibration temperature on validation before the one final test evaluation.

In Colab choose **Runtime → Change runtime type → T4 GPU**. Upload this repository to `/content/loraforge-llm` (or clone it if you later host it privately and authenticate).

In [ ]:
%cd /content/loraforge-llm
!python -m pip install -q -e ".[gpu]"

In [ ]:
import json, platform, time
from pathlib import Path
import numpy as np
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name, f'Frozen evidence run expects a T4, found {gpu_name}'
print({'gpu': gpu_name, 'torch': torch.__version__, 'python': platform.python_version()})
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!python -m pytest -q

In [ ]:
from loraforge.config import default_config
from loraforge.data import describe, load_dataset
from loraforge.prompts import training_messages

config = default_config()
bundle = load_dataset(allow_test=False, config=config.data)
assert bundle.test is None
print(json.dumps(describe(bundle, config.data), indent=2))
for item in bundle.train.examples[:3]:
    print(json.dumps(training_messages(item.text, item.label), indent=2))

## Untuned validation baseline

The base and tuned models use the same chat prompt and the same conditional A–D class logits. Temperature is fit on validation only. This cell does not read test data.

In [ ]:
from loraforge.modeling import load_quantized_base
from loraforge.prompts import class_code_token_ids

torch.manual_seed(config.data.seed)
model, tokenizer = load_quantized_base(config)
print('contextual class token IDs:', class_code_token_ids(tokenizer))
print('allocated GiB after load:', torch.cuda.memory_allocated() / 1024**3)

In [ ]:
from loraforge.evaluation import validation_baseline, write_report

started = time.perf_counter()
base_validation = validation_baseline(model, tokenizer, bundle, config)
base_validation['wall_time_seconds'] = time.perf_counter() - started
base_validation['gpu_name'] = gpu_name
write_report(base_validation, Path('outputs/base-validation.json'))
print(json.dumps({
    'macro_f1': base_validation['metrics_before_temperature']['macro_f1'],
    'ece_before': base_validation['metrics_before_temperature']['calibration']['ece'],
    'ece_after': base_validation['metrics_after_temperature']['calibration']['ece'],
    'temperature': base_validation['validation_temperature'],
    'test_evaluated': base_validation['test_evaluated'],
}, indent=2))

## Attach QLoRA adapters and audit trainability

This proves the setup, but it does not claim the adapter fits a T4 until this cell actually completes and writes its hardware evidence.

In [ ]:
from datetime import UTC, datetime
from loraforge.qlora import attach_lora, parameter_report

model = attach_lora(model, config)
parameters = parameter_report(model)
setup = {
    'schema_version': 1,
    'created_at_utc': datetime.now(UTC).isoformat().replace('+00:00', 'Z'),
    'gpu_name': gpu_name,
    'model': config.model_name,
    'model_revision': config.model_revision,
    'quantization': config.to_dict()['quantization'],
    'lora': config.to_dict()['lora'],
    'parameters': parameters,
    'allocated_cuda_gib': torch.cuda.memory_allocated() / 1024**3,
    'peak_cuda_gib': torch.cuda.max_memory_allocated() / 1024**3,
    'test_loaded': False,
    'adapter_trained': False,
}
Path('outputs').mkdir(exist_ok=True)
Path('outputs/qlora-setup.json').write_text(json.dumps(setup, indent=2) + '\n')
print(json.dumps(setup, indent=2))

# STOP — Claude handoff begins here

Download `outputs/base-validation.json` and `outputs/qlora-setup.json`, place them back in the repository, and give Claude Code `docs/CLAUDE_HANDOFF.md`. Claude owns training, validation-only selection, the single base-vs-tuned publisher-test run, final calibration/reporting, hardening, and publication. Do not improvise a test run from this notebook.